# Moduł 8: Sieci neuronowe MLP w PyTorch

**MLP (Multi-Layer Perceptron)** — sieć neuronowa z warstwami w pełni połączonymi.

## Spis treści
1. Sztuczny neuron
2. Funkcje aktywacji
3. MLP — architektura
4. Forward & backward pass
5. Trening na MNIST w PyTorch
6. Techniki regularyzacji
7. Inicjalizacja wag (Weight Initialization)
8. Autoenkoery
9. **Zadania**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

## 1. Sztuczny neuron (perceptron)

Neuron oblicza **ważoną sumę** wejść + bias, a następnie przepuszcza przez **funkcję aktywacji**:

$$z = \sum_{i=1}^{n} w_i \cdot x_i + b = \mathbf{w}^T \mathbf{x} + b$$

$$a = \sigma(z)$$

gdzie $\sigma$ to funkcja aktywacji (sigmoid, ReLU, tanh...)

### Warstwa w pełni połączona (fully connected / dense)

Dla warstwy z $m$ neuronami i $n$ wejściami:

$$\mathbf{z} = W \mathbf{x} + \mathbf{b}$$

gdzie $W \in \mathbb{R}^{m \times n}$, $\mathbf{b} \in \mathbb{R}^m$.

Liczba parametrów warstwy: $m \cdot n + m = m(n + 1)$

### Dlaczego potrzebujemy aktywacji?
Bez nieliniowej aktywacji, złożenie wielu warstw liniowych = jedna warstwa liniowa:

$$W_2(W_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2 = W_2 W_1 \mathbf{x} + (W_2\mathbf{b}_1 + \mathbf{b}_2) = W' \mathbf{x} + \mathbf{b}'$$

Aktywacja dodaje **nieliniowość**, dzięki której sieć może aproksymować dowolną funkcję!

### Twierdzenie o uniwersalnej aproksymacji (Cybenko, 1989)

MLP z **jedną** ukrytą warstwą z nieliniową aktywacją i **wystarczającą liczbą neuronów** może aproksymować dowolną ciągłą funkcję $f: \mathbb{R}^n \to \mathbb{R}$ na zbiorze zwartym z dowolną dokładnością $\epsilon > 0$.

## 2. Funkcje aktywacji

| Funkcja | Wzór | Pochodna | Zakres | Użycie |
|---------|------|----------|--------|--------|
| **Sigmoid** | $\sigma(x) = \frac{1}{1+e^{-x}}$ | $\sigma(x)(1-\sigma(x))$ | (0, 1) | Wyjście binarne |
| **Tanh** | $\tanh(x) = \frac{e^x-e^{-x}}{e^x+e^{-x}}$ | $1 - \tanh^2(x)$ | (-1, 1) | Ukryte warstwy (rzadko) |
| **ReLU** | $\max(0, x)$ | $\begin{cases}1 & x > 0 \\ 0 & x \leq 0\end{cases}$ | $[0, \infty)$ | **Domyślna** |
| **Leaky ReLU** | $\max(\alpha x, x)$ | $\begin{cases}1 & x > 0 \\ \alpha & x \leq 0\end{cases}$ | $(-\infty, \infty)$ | Unika "dying ReLU" |
| **GELU** | $x \cdot \Phi(x)$ | — | $(-\infty, \infty)$ | Transformery |
| **Softmax** | $\frac{e^{x_i}}{\sum_j e^{x_j}}$ | — | (0,1), $\sum=1$ | Wyjście wieloklasowe |

### Problem zanikającego gradientu (Vanishing Gradient)

Sigmoid i tanh mają **nasycone** regiony (gradient $\approx 0$ dla dużych $|x|$):
- Sigmoid: $\sigma'(x) \leq 0.25$ (max w $x = 0$)
- Gradient maleje wykładniczo przez kolejne warstwy: $\frac{\partial L}{\partial w_1} \propto \prod_{l} \sigma'(z_l)$

**ReLU** rozwiązuje ten problem: gradient = 1 dla $x > 0$, ale ma problem "dying neurons" (gradient = 0 dla $x < 0$ → neuron nigdy się nie aktywuje).

### Softmax — szczegóły

Dla wektora logitów $\mathbf{z} = [z_1, \ldots, z_C]$:

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{C} e^{z_j}} = P(\text{klasa} = i | \mathbf{x})$$

**Trick numeryczny**: odejmij $\max(\mathbf{z})$ przed exp, aby uniknąć overflow:

$$\text{softmax}(z_i) = \frac{e^{z_i - \max(\mathbf{z})}}{\sum_j e^{z_j - \max(\mathbf{z})}}$$

In [ ]:
# Wizualizacja funkcji aktywacji
x = np.linspace(-5, 5, 200)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Sigmoid
axes[0].plot(x, 1 / (1 + np.exp(-x)), 'b-', linewidth=2)
axes[0].set_title("Sigmoid")
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)

# Tanh
axes[1].plot(x, np.tanh(x), 'r-', linewidth=2)
axes[1].set_title("Tanh")
axes[1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# ReLU
axes[2].plot(x, np.maximum(0, x), 'g-', linewidth=2)
axes[2].set_title("ReLU")

# Leaky ReLU
axes[3].plot(x, np.where(x > 0, x, 0.01 * x), 'm-', linewidth=2)
axes[3].set_title("Leaky ReLU")

for ax in axes:
 ax.grid(True, alpha=0.3)
 ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 3. MLP — architektura

```
Wejście (784) → Warstwa ukryta 1 (256, ReLU) → Warstwa ukryta 2 (128, ReLU) → Wyjście (10, Softmax)
```

### Warstwy w PyTorch
- `nn.Linear(in, out)` — warstwa w pełni połączona (fully connected)
- `nn.ReLU()` — aktywacja
- `nn.Dropout(p)` — losowo wyłącza p% neuronów (regularyzacja)
- `nn.BatchNorm1d(features)` — normalizacja per batch

### Liczba parametrów

Dla powyższej architektury:
- Warstwa 1: $784 \times 256 + 256 = 200{,}960$
- Warstwa 2: $256 \times 128 + 128 = 32{,}896$
- Warstwa 3: $128 \times 10 + 10 = 1{,}290$
- **Razem: 235,146 parametrów**

### Cross-Entropy Loss — funkcja straty dla klasyfikacji

Dla prawdziwej klasy $c$ i predykcji softmax $p_i$:

$$\mathcal{L}_{\text{CE}} = -\log(p_c) = -\log\left(\frac{e^{z_c}}{\sum_j e^{z_j}}\right)$$

**W PyTorch** `nn.CrossEntropyLoss` łączy softmax + NLL Loss (Negative Log Likelihood):

$$\mathcal{L}_{\text{CE}} = -z_c + \log\sum_j e^{z_j}$$

Dlatego **NIE** dodawaj softmax do ostatniej warstwy, jeśli używasz `CrossEntropyLoss`!

### Binary Cross-Entropy

Dla klasyfikacji binarnej ($y \in \{0, 1\}$, predykcja $p$):

$$\mathcal{L}_{\text{BCE}} = -[y \log p + (1-y) \log(1-p)]$$

### Dropout — regularyzacja w sieciach neuronowych

Podczas treningu losowo zerujemy $p$ frakcję neuronów. Podczas inferencji skalujemy wyjścia przez $(1-p)$:

$$\text{train:} \quad \tilde{a}_i = a_i \cdot r_i, \quad r_i \sim \text{Bernoulli}(1-p)$$

$$\text{test:} \quad \tilde{a}_i = (1-p) \cdot a_i$$

**Intuicja**: każdy neuron nie może polegać na żadnym konkretnym innym neuronie → wymusza redundancję.

### Batch Normalization

Normalizacja aktywacji per mini-batch:

$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}, \quad y_i = \gamma \hat{x}_i + \beta$$

gdzie $\mu_B, \sigma_B^2$ to średnia i wariancja mini-batcha, a $\gamma, \beta$ to uczalne parametry.

**Efekt:** stabilizuje trening, pozwala na wyższy learning rate, działa lekko regularyzująco.

### Dlaczego MNIST?
MNIST to 70 000 ręcznie pisanych cyfr (28×28 pikseli, 10 klas).
To "Hello World" deep learningu!

In [ ]:
# Ładowanie MNIST
transform = transforms.Compose([
 transforms.ToTensor(),
 transforms.Normalize((0.1307,), (0.3081,)) # średnia i std MNIST
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Podział train na train + val
train_set, val_set = random_split(train_dataset, [50000, 10000])

# DataLoadery
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_dataset)}")

# Podejrzyjmy dane
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
 img, label = train_dataset[i]
 ax.imshow(img.squeeze(), cmap='gray')
 ax.set_title(f"Label: {label}")
 ax.axis('off')
plt.suptitle("Przykładowe cyfry MNIST")
plt.show()

## 4. Forward & backward pass

### Forward pass
Dane przepływają od wejścia do wyjścia przez kolejne warstwy:

$$\mathbf{x} \xrightarrow{W_1, b_1} \mathbf{z}_1 \xrightarrow{\sigma} \mathbf{a}_1 \xrightarrow{W_2, b_2} \mathbf{z}_2 \xrightarrow{\sigma} \mathbf{a}_2 \xrightarrow{W_3, b_3} \mathbf{z}_3 \xrightarrow{\text{softmax}} \hat{\mathbf{y}}$$

### Backward pass (backpropagation)

Gradient błędu propaguje się **wstecz** od wyjścia do wejścia, używając **reguły łańcuchowej** (chain rule):

$$\frac{\partial \mathcal{L}}{\partial w_1} = \frac{\partial \mathcal{L}}{\partial a_3} \cdot \frac{\partial a_3}{\partial z_3} \cdot \frac{\partial z_3}{\partial a_2} \cdot \frac{\partial a_2}{\partial z_2} \cdot \frac{\partial z_2}{\partial w_1}$$

### Gradienty dla poszczególnych operacji

Dla warstwy liniowej $\mathbf{z} = W\mathbf{a} + \mathbf{b}$:

$$\frac{\partial \mathcal{L}}{\partial W} = \frac{\partial \mathcal{L}}{\partial \mathbf{z}} \cdot \mathbf{a}^T, \quad \frac{\partial \mathcal{L}}{\partial \mathbf{b}} = \frac{\partial \mathcal{L}}{\partial \mathbf{z}}, \quad \frac{\partial \mathcal{L}}{\partial \mathbf{a}} = W^T \frac{\partial \mathcal{L}}{\partial \mathbf{z}}$$

Dla ReLU $\mathbf{a} = \text{ReLU}(\mathbf{z})$:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{z}} = \frac{\partial \mathcal{L}}{\partial \mathbf{a}} \odot \mathbb{1}[\mathbf{z} > 0]$$

### Gradient Softmax + Cross-Entropy (razem!)

Piękny wynik — gradient jest po prostu:

$$\frac{\partial \mathcal{L}_{\text{CE}}}{\partial z_i} = p_i - y_i$$

gdzie $p_i = \text{softmax}(z_i)$ a $y_i$ to one-hot encoding klasy.

**PyTorch oblicza to automatycznie** za pomocą `loss.backward()`!

## 5. Trening MLP na MNIST

In [ ]:
# Definicja modelu MLP
class MLP(nn.Module):
 def __init__(self):
 super().__init__()
 self.flatten = nn.Flatten() # 28x28 → 784
 self.layers = nn.Sequential(
 nn.Linear(784, 256),
 nn.ReLU(),
 nn.Dropout(0.2), # regularyzacja
 nn.Linear(256, 128),
 nn.ReLU(),
 nn.Dropout(0.2),
 nn.Linear(128, 10) # 10 klas
 )

 def forward(self, x):
 x = self.flatten(x)
 return self.layers(x)

model = MLP()
print(model)
print(f"\nLiczba parametrów: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Definicja treningu
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = MLP().to(device)
criterion = nn.CrossEntropyLoss() # loss dla klasyfikacji wieloklasowej
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_epoch(model, loader, criterion, optimizer, device):
 model.train() # tryb treningowy (dropout aktywny)
 total_loss, correct, total = 0, 0, 0
 for X_batch, y_batch in loader:
 X_batch, y_batch = X_batch.to(device), y_batch.to(device)

 optimizer.zero_grad()
 output = model(X_batch)
 loss = criterion(output, y_batch)
 loss.backward()
 optimizer.step()

 total_loss += loss.item() * X_batch.size(0)
 correct += (output.argmax(1) == y_batch).sum().item()
 total += X_batch.size(0)
 return total_loss / total, correct / total

def evaluate(model, loader, criterion, device):
 model.eval() # tryb ewaluacji (dropout wyłączony)
 total_loss, correct, total = 0, 0, 0
 with torch.no_grad(): # nie liczymy gradientów
 for X_batch, y_batch in loader:
 X_batch, y_batch = X_batch.to(device), y_batch.to(device)
 output = model(X_batch)
 loss = criterion(output, y_batch)
 total_loss += loss.item() * X_batch.size(0)
 correct += (output.argmax(1) == y_batch).sum().item()
 total += X_batch.size(0)
 return total_loss / total, correct / total

In [ ]:
# Trening!
num_epochs = 10
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(num_epochs):
 train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
 val_loss, val_acc = evaluate(model, val_loader, criterion, device)

 history["train_loss"].append(train_loss)
 history["val_loss"].append(val_loss)
 history["train_acc"].append(train_acc)
 history["val_acc"].append(val_acc)

 print(f"Epoch {epoch+1:2d}/{num_epochs}: "
 f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
 f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

# Ewaluacja na teście
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"\n>>> Test accuracy: {test_acc:.4f}")

In [ ]:
# Krzywe uczenia
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["train_acc"], label="Train")
axes[1].plot(history["val_acc"], label="Val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.suptitle("MLP na MNIST")
plt.tight_layout()
plt.show()

## 6. Techniki regularyzacji

| Technika | Opis |
|----------|------|
| **Dropout** | Losowo wyłącza neurony podczas treningu (p=0.2-0.5) |
| **Weight decay** | L2 regularyzacja wag (parametr w optimizerze) |
| **Early stopping** | Zatrzymaj trening gdy val_loss rośnie |
| **Batch normalization** | Normalizacja aktywacji w mini-batchu |
| **Data augmentation** | Zwiększenie zbioru treningowego przez transformacje |

## 7. Inicjalizacja wag (Weight Initialization)

**Zła inicjalizacja** → gradienty zanikają lub eksplodują, sieć się nie uczy!

### Problem

Jeśli wagi są:
- **Za małe** → aktywacje maleją warstwa po warstwie → **vanishing gradients**
- **Za duże** → aktywacje eksplodują → **exploding gradients / NaN**
- **Identyczne** (np. same zera) → wszystkie neurony uczą się tego samego (**symmetry problem**)

### Cel: zachować wariancję aktywacji

Chcemy, by wariancja wyjścia warstwy $\approx$ wariancji wejścia: $\text{Var}(a_l) \approx \text{Var}(a_{l-1})$

### Xavier / Glorot Initialization (2010)

Dla aktywacji **Sigmoid / Tanh**:

$$W \sim \mathcal{N}\left(0, \frac{2}{n_{\text{in}} + n_{\text{out}}}\right) \quad \text{lub} \quad W \sim \mathcal{U}\left(-\sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}}, \sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}}\right)$$

- $n_{\text{in}}$ = liczba neuronów wejściowych (fan\_in)
- $n_{\text{out}}$ = liczba neuronów wyjściowych (fan\_out)
- Równoważy wariancję przy forward **i** backward pass

### Kaiming / He Initialization (2015)

Dla aktywacji **ReLU** (i wariantów):

$$W \sim \mathcal{N}\left(0, \frac{2}{n_{\text{in}}}\right)$$

**Dlaczego $\times 2$?** ReLU zeruje ~50% aktywacji, więc potrzebna jest większa wariancja, by kompensować utratę.

### Podsumowanie

| Inicjalizacja | Aktywacja | PyTorch |
|--------------|-----------|---------|
| **Xavier uniform** | Sigmoid, Tanh | `nn.init.xavier_uniform_(w)` |
| **Xavier normal** | Sigmoid, Tanh | `nn.init.xavier_normal_(w)` |
| **Kaiming uniform** | ReLU, LeakyReLU | `nn.init.kaiming_uniform_(w, mode='fan_in', nonlinearity='relu')` |
| **Kaiming normal** | ReLU, LeakyReLU | `nn.init.kaiming_normal_(w, mode='fan_in', nonlinearity='relu')` |
| **Stałe** | Biasy | `nn.init.zeros_(b)` lub `nn.init.constant_(b, 0)` |
| **Ortogonalna** | RNN | `nn.init.orthogonal_(w)` |

**Reguła kciuka:**
- ReLU → **Kaiming** (He)
- Sigmoid/Tanh → **Xavier** (Glorot)
- `nn.Linear` w PyTorch domyślnie używa **Kaiming uniform**

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

# =========================================
# Porównanie inicjalizacji wag
# =========================================
def test_init(init_fn, name, num_layers=10, dim=256, activation=nn.ReLU()):
 """Sprawdza jak wariancja aktywacji zmienia się warstwa po warstwie."""
 x = torch.randn(100, dim) # batch=100
 variances = [x.var().item()]

 for _ in range(num_layers):
 layer = nn.Linear(dim, dim, bias=False)
 init_fn(layer.weight)
 with torch.no_grad():
 x = activation(layer(x))
 variances.append(x.var().item())

 return variances

# Testujemy różne inicjalizacje z ReLU
inits = {
 "Zeros (ZŁE!)": lambda w: nn.init.zeros_(w),
 "Random N(0,1) (ZŁE!)": lambda w: nn.init.normal_(w, 0, 1),
 "Random N(0,0.01) (ZŁE!)": lambda w: nn.init.normal_(w, 0, 0.01),
 "Xavier uniform": lambda w: nn.init.xavier_uniform_(w),
 "Kaiming normal ✓": lambda w: nn.init.kaiming_normal_(w, nonlinearity='relu'),
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, init_fn in inits.items():
 vars_ = test_init(init_fn, name)
 axes[0].plot(vars_, label=name, marker='o', markersize=4)

axes[0].set_xlabel("Warstwa")
axes[0].set_ylabel("Wariancja aktywacji")
axes[0].set_title("Wariancja aktywacji vs głębokość (ReLU)")
axes[0].legend(fontsize=8)
axes[0].set_yscale("log")

# Porównanie Xavier vs Kaiming z odpowiednimi aktywacjami
for name, init_fn, act in [
 ("Xavier + Tanh ✓", lambda w: nn.init.xavier_uniform_(w), nn.Tanh()),
 ("Kaiming + Tanh ✗", lambda w: nn.init.kaiming_normal_(w, nonlinearity='relu'), nn.Tanh()),
 ("Xavier + ReLU ✗", lambda w: nn.init.xavier_uniform_(w), nn.ReLU()),
 ("Kaiming + ReLU ✓", lambda w: nn.init.kaiming_normal_(w, nonlinearity='relu'), nn.ReLU()),
]:
 vars_ = test_init(init_fn, name, activation=act)
 axes[1].plot(vars_, label=name, marker='o', markersize=4)

axes[1].set_xlabel("Warstwa")
axes[1].set_ylabel("Wariancja aktywacji")
axes[1].set_title("Dopasowanie inicjalizacji do aktywacji")
axes[1].legend(fontsize=8)
axes[1].set_yscale("log")

plt.tight_layout()
plt.show()

# =========================================
# Jak inicjalizować wagi w praktyce
# =========================================
class MLPWithInit(nn.Module):
 def __init__(self, input_dim, hidden_dim, output_dim):
 super().__init__()
 self.fc1 = nn.Linear(input_dim, hidden_dim)
 self.fc2 = nn.Linear(hidden_dim, hidden_dim)
 self.fc3 = nn.Linear(hidden_dim, output_dim)
 self.relu = nn.ReLU()

 # Ręczna inicjalizacja Kaiming (dla ReLU)
 self._init_weights()

 def _init_weights(self):
 for m in self.modules():
 if isinstance(m, nn.Linear):
 nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
 nn.init.zeros_(m.bias)

 def forward(self, x):
 x = self.relu(self.fc1(x))
 x = self.relu(self.fc2(x))
 return self.fc3(x)

model = MLPWithInit(784, 256, 10)
print("Model z Kaiming init:")
for name, param in model.named_parameters():
 if 'weight' in name:
 print(f" {name}: mean={param.data.mean():.4f}, std={param.data.std():.4f}")

## 8. Autoenkoery

**Autoenkoder** to sieć neuronowa uczona rekonstruować swoje wejście, przechodząc przez **wąskie gardło** (bottleneck).

### Architektura

```
Wejście x (784) Wyjście x̂ (784)
 ↓ ↑
 Encoder Decoder
 ↓ ↑
┌─────────┐ ┌──────────┐ ┌──────────┐
│ FC(784→256) → │ FC(256→32) → │ FC(32→256) → FC(256→784)
│ │ │ Latent z │ │ │
└─────────┘ └──────────┘ └──────────┘
```

### Wzory

**Encoder:** $\mathbf{z} = f_\theta(\mathbf{x})$ — mapuje wejście na reprezentację latentną

**Decoder:** $\hat{\mathbf{x}} = g_\phi(\mathbf{z})$ — rekonstruuje wejście z reprezentacji

**Reconstruction Loss:** 
$$\mathcal{L}_{\text{recon}} = \|\mathbf{x} - \hat{\mathbf{x}}\|^2 = \text{MSE}(\mathbf{x}, \hat{\mathbf{x}})$$

lub Binary Cross-Entropy (dla danych znormalizowanych do $[0,1]$):
$$\mathcal{L}_{\text{BCE}} = -\sum_i [x_i \log \hat{x}_i + (1-x_i)\log(1-\hat{x}_i)]$$

### Rodzaje autoenkoerów

| Typ | Opis | Zastosowanie |
|-----|------|-------------|
| **Vanilla AE** | Prosty encoder-decoder, bottleneck < input | Kompresja, denoising |
| **Denoising AE** | Dodaj szum do wejścia, rekonstruuj oryginał | Odszumianie, lepsze cechy |
| **Sparse AE** | Regularyzacja: aktywacje bottleneck powinny być rzadkie | Feature learning |
| **Variational AE (VAE)** | Latent space to rozkład $\mathcal{N}(\mu, \sigma^2)$ | Generowanie nowych danych |
| **Convolutional AE** | Conv layers zamiast FC | Obrazy |

### VAE (Variational Autoencoder)

W VAE encoder produkuje **parametry rozkładu** $\mu, \sigma$ zamiast deterministycznego $z$:

$$z \sim \mathcal{N}(\mu, \sigma^2) \quad \text{(reparametrization trick: } z = \mu + \sigma \cdot \epsilon, \; \epsilon \sim \mathcal{N}(0,1)\text{)}$$

$$\mathcal{L}_{\text{VAE}} = \underbrace{\mathcal{L}_{\text{recon}}}_{\text{rekonstrukcja}} + \underbrace{D_{\text{KL}}(\mathcal{N}(\mu, \sigma^2) \| \mathcal{N}(0, I))}_{\text{regularyzacja latent space}}$$

$$D_{\text{KL}} = -\frac{1}{2}\sum_{j=1}^{d}(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2)$$

### Zastosowania

- **Redukcja wymiarowości** (alternatywa dla PCA — nieliniowa)
- **Denoising** (odszumianie obrazów, dźwięku)
- **Anomaly detection** (wysoki reconstruction error = anomalia)
- **Generowanie** danych (VAE)
- **Feature extraction** (bottleneck = embedding)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# =========================================
# Autoenkoder na MNIST
# =========================================
class Autoencoder(nn.Module):
 def __init__(self, latent_dim=32):
 super().__init__()
 # Encoder: 784 → 256 → latent_dim
 self.encoder = nn.Sequential(
 nn.Linear(784, 256),
 nn.ReLU(),
 nn.Linear(256, latent_dim),
 nn.ReLU(),
 )
 # Decoder: latent_dim → 256 → 784
 self.decoder = nn.Sequential(
 nn.Linear(latent_dim, 256),
 nn.ReLU(),
 nn.Linear(256, 784),
 nn.Sigmoid(), # wyjście w [0, 1]
 )

 def forward(self, x):
 z = self.encoder(x) # encoding
 x_hat = self.decoder(z) # reconstruction
 return x_hat, z

# Przygotowanie danych
transform = transforms.Compose([transforms.ToTensor()])
train_data = datasets.MNIST("./data", train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=256, shuffle=True)

# Trening
model = Autoencoder(latent_dim=32)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

print("Trening Autoencodera na MNIST...")
for epoch in range(5):
 total_loss = 0
 for images, _ in train_loader:
 x = images.view(-1, 784)
 x_hat, z = model(x)
 loss = criterion(x_hat, x)

 optimizer.zero_grad()
 loss.backward()
 optimizer.step()
 total_loss += loss.item()

 avg_loss = total_loss / len(train_loader)
 print(f" Epoch {epoch+1}/5, Loss: {avg_loss:.4f}")

# Wizualizacja rekonstrukcji
model.eval()
with torch.no_grad():
 test_data = datasets.MNIST("./data", train=False, transform=transform)
 test_images = torch.stack([test_data[i][0] for i in range(10)])
 x = test_images.view(-1, 784)
 x_hat, z = model(x)

fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
 axes[0, i].imshow(test_images[i].squeeze(), cmap='gray')
 axes[0, i].axis('off')
 axes[1, i].imshow(x_hat[i].view(28, 28).numpy(), cmap='gray')
 axes[1, i].axis('off')
axes[0, 0].set_ylabel("Oryginał", fontsize=12)
axes[1, 0].set_ylabel("Rekonstrukcja", fontsize=12)
plt.suptitle(f"Autoenkoder (latent_dim=32), MSE={avg_loss:.4f}")
plt.tight_layout()
plt.show()

# =========================================
# Anomaly Detection z AE
# =========================================
print("\n--- Anomaly Detection ---")
with torch.no_grad():
 # Normalne cyfry (0-9)
 normal_errors = []
 for i in range(100):
 x = test_data[i][0].view(1, 784)
 x_hat, _ = model(x)
 err = ((x - x_hat) ** 2).mean().item()
 normal_errors.append(err)

 # "Anomalny" obraz — losowy szum
 noise = torch.rand(1, 784)
 noise_hat, _ = model(noise)
 noise_err = ((noise - noise_hat) ** 2).mean().item()

print(f"Średni error (normalne): {sum(normal_errors)/len(normal_errors):.4f}")
print(f"Error (losowy szum): {noise_err:.4f}")
print(f"→ Wysoki reconstruction error = anomalia!")

---
---
## Zadania do samodzielnego rozwiązania

---

### Zadanie 1: Architektura MLP (łatwe)

Zmodyfikuj architekturę MLP:
- Dodaj trzecią warstwę ukrytą (64 neurony)
- Zmień aktywację na LeakyReLU
- Dodaj BatchNorm po każdej warstwie liniowej

Porównaj wynik z oryginalnym MLP.

In [ ]:
# Zadanie 1
class MLP_v2(nn.Module):
 def __init__(self):
 super().__init__()
 # TWÓJ KOD TUTAJ
 pass

 def forward(self, x):
 pass

### Zadanie 2: Early stopping (średnie)

Zmodyfikuj pętlę treningową:
1. Śledź najlepszy `val_loss`
2. Jeśli `val_loss` nie poprawia się przez `patience=3` epoki → zatrzymaj trening
3. Zapisz najlepszy model (`torch.save`)
4. Po treningu wczytaj najlepszy model i oceń na teście

In [ ]:
# Zadanie 2: Early stopping
# TWÓJ KOD TUTAJ

### Zadanie 3: Klasyfikacja Fashion-MNIST (trudne)

Fashion-MNIST to 28×28 obrazów ubrań (10 klas: T-shirt, spodnie, buty...).

1. Załaduj `datasets.FashionMNIST`
2. Zaprojektuj MLP z co najmniej 3 warstwami ukrytymi
3. Użyj: BatchNorm, Dropout, Adam, scheduler (ReduceLROnPlateau)
4. Trenuj 15 epok z early stopping
5. Narysuj confusion matrix
6. Pokaż przykłady błędnie sklasyfikowanych obrazów

In [ ]:
# Zadanie 3: Fashion-MNIST
# TWÓJ KOD TUTAJ

### Zadanie 4: XOR — nieliniowość (średnie)

Problem XOR nie jest liniowo separowalny!

| x1 | x2 | y |
|----|----|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

1. Pokaż, że `nn.Linear(2, 1)` (bez ukrytej warstwy) NIE radzi sobie z XOR
2. Dodaj warstwę ukrytą (np. 4 neurony + ReLU) i pokaż, że teraz działa
3. Wizualizuj granicę decyzyjną obu modeli

In [ ]:
# Zadanie 4: XOR
# TWÓJ KOD TUTAJ

### Zadanie 5: Efekt dropout (srednie)

Na zbiorze MNIST:
1. Wytrenuj MLP **bez dropout** (3 warstwy, 256 neuronow)
2. Wytrenuj identyczny MLP **z dropout** (p=0.3, p=0.5)
3. Porownaj krzywe train loss i val loss na jednym wykresie
4. Kiedy dropout pomaga, a kiedy szkodzi? Zaobserwuj i opisz.

In [ ]:
# Zadanie 5: Efekt dropout
import torch
import torch.nn as nn
torch.manual_seed(42)
# TWOJ KOD TUTAJ

### Zadanie 6: Porownanie inicjalizacji wag (trudne)

Zbuduj MLP (5 warstw, 512 neuronow, ReLU) i przetestuj 4 metody inicjalizacji:
1. `torch.nn.init.zeros_` (zerowa)
2. `torch.nn.init.normal_` (losowa, std=1.0)
3. `torch.nn.init.kaiming_normal_` (He)
4. `torch.nn.init.xavier_normal_` (Glorot)

Dla kazdej metody:
- Przepusc losowe dane przez siec (forward pass)
- Wypisz srednia i std aktywacji po kazdej warstwie
- Ktora inicjalizacja zachowuje stabilne aktywacje?

In [ ]:
# Zadanie 6: Inicjalizacja wag
import torch
import torch.nn as nn
torch.manual_seed(42)
# TWOJ KOD TUTAJ

---

## Dodatek OAI: Ćwiczenia w stylu olimpiadowym

### Kontekst olimpiadowy
Sieci MLP i PyTorch to **serce** większości zadań OAI. Zadania z poprzednich edycji:

- **I OAI:** Pruning — przycinanie wag w sieci neuronowej (moduł 08!)
- **II OAI, Etap 1:** Noisy labels — trenowanie z zaszumionymi etykietami (stała architektura SmallMobileNet)
- **II OAI, Etap 2:** Rozkład nienormalny — denoising autoencoder + klasyfikacja
- **II OAI, Finał:** Inpainting z INR (Implicit Neural Representations) — MLP mapujący (x,y) → kolor
- **II OAI, Finał:** Machine unlearning — usunięcie klasy z modelu bez pełnego retrenowania

### Ćwiczenie OAI-8A: Pruning sieci neuronowej

Wytrenuj MLP na MNIST, a potem przytnij 50% wag (wyzeruj najniższe) i zmierz spadek accuracy. Następnie dotrenuj (fine-tune) pruned model.

### Ćwiczenie OAI-8B: Trenowanie z zaszumionymi etykietami

Stwórz zbiór z 20% losowo zmienionych etykiet. Porównaj accuracy modelu trenowanego na czystych vs zaszumionych danych. Przetestuj techniki odporne na szum (np. Label Smoothing).

### Ćwiczenie OAI-8C: Implicit Neural Representation (INR)

Stwórz MLP, który mapuje współrzędne pikseli (x, y) → wartość piksela (0-1). Wytrenuj go na prostym obrazku 32x32 i zrekonstruuj obraz z modelu.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# === OAI-8A: Pruning sieci neuronowej ===
print("=== Pruning (wzorzec: I OAI) ===")

# Prosty MLP
class SimpleMLP(nn.Module):
 def __init__(self):
 super().__init__()
 self.fc1 = nn.Linear(784, 256)
 self.fc2 = nn.Linear(256, 128)
 self.fc3 = nn.Linear(128, 10)
 self.relu = nn.ReLU()

 def forward(self, x):
 x = x.view(-1, 784)
 x = self.relu(self.fc1(x))
 x = self.relu(self.fc2(x))
 return self.fc3(x)

# Funkcja pruning — wyzeruj wagi o najniższej wartości bezwzględnej
def prune_model(model, prune_ratio=0.5):
 """Przytnij prune_ratio% wag (wyzeruj najniższe)."""
 total_params = 0
 pruned_params = 0

 for name, param in model.named_parameters():
 if 'weight' in name:
 tensor = param.data.abs()
 threshold = torch.quantile(tensor.flatten(), prune_ratio)
 mask = tensor > threshold
 param.data *= mask.float()
 total_params += param.numel()
 pruned_params += (mask == 0).sum().item()

 print(f" Przycięto {pruned_params}/{total_params} wag ({100*pruned_params/total_params:.1f}%)")
 return model

# Demonstracja (bez trenowania — tylko struktura)
model = SimpleMLP()
print(f" Parametry modelu: {sum(p.numel() for p in model.parameters()):,}")
model_pruned = prune_model(model, prune_ratio=0.5)

# === OAI-8B: Label Smoothing ===
print("\n=== Label Smoothing (technika na noisy labels, II OAI) ===")

class LabelSmoothingLoss(nn.Module):
 def __init__(self, num_classes=10, smoothing=0.1):
 super().__init__()
 self.smoothing = smoothing
 self.num_classes = num_classes

 def forward(self, pred, target):
 confidence = 1.0 - self.smoothing
 smooth_val = self.smoothing / (self.num_classes - 1)

 one_hot = torch.full_like(pred, smooth_val)
 one_hot.scatter_(1, target.unsqueeze(1), confidence)

 log_probs = torch.log_softmax(pred, dim=1)
 loss = -(one_hot * log_probs).sum(dim=1).mean()
 return loss

# Porównanie
dummy_pred = torch.randn(4, 10)
dummy_target = torch.tensor([0, 3, 7, 1])
print(f" CrossEntropy: {nn.CrossEntropyLoss()(dummy_pred, dummy_target):.4f}")
print(f" LabelSmoothing: {LabelSmoothingLoss(smoothing=0.1)(dummy_pred, dummy_target):.4f}")

# === OAI-8C: Implicit Neural Representation ===
print("\n=== INR — Implicit Neural Representation (wzorzec: Inpainting, II OAI finał) ===")

class INR(nn.Module):
 """MLP mapujący (x, y) → wartość piksela."""
 def __init__(self, hidden_dim=128):
 super().__init__()
 self.net = nn.Sequential(
 nn.Linear(2, hidden_dim),
 nn.ReLU(),
 nn.Linear(hidden_dim, hidden_dim),
 nn.ReLU(),
 nn.Linear(hidden_dim, hidden_dim),
 nn.ReLU(),
 nn.Linear(hidden_dim, 1),
 nn.Sigmoid()
 )

 def forward(self, coords):
 return self.net(coords)

# Stwórz prosty obraz — gradient
size = 32
x = torch.linspace(0, 1, size)
y = torch.linspace(0, 1, size)
grid_x, grid_y = torch.meshgrid(x, y, indexing='ij')
coords = torch.stack([grid_x.flatten(), grid_y.flatten()], dim=1)
# Obraz: kółko
center = torch.tensor([0.5, 0.5])
target_img = (torch.norm(coords - center, dim=1, keepdim=True) < 0.3).float()

# Trenuj INR
inr = INR(hidden_dim=64)
optimizer = optim.Adam(inr.parameters(), lr=1e-3)
for epoch in range(500):
 pred = inr(coords)
 loss = nn.MSELoss()(pred, target_img)
 optimizer.zero_grad()
 loss.backward()
 optimizer.step()

# Rekonstrukcja
with torch.no_grad():
 reconstructed = inr(coords).reshape(size, size).numpy()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(target_img.reshape(size, size).numpy(), cmap='gray')
axes[0].set_title("Oryginał")
axes[1].imshow(reconstructed, cmap='gray')
axes[1].set_title(f"INR (loss={loss.item():.4f})")
plt.suptitle("INR — Implicit Neural Representation\n(Wzorzec: Inpainting, II OAI finał)")
plt.tight_layout()
plt.show()

psnr = 20 * np.log10(1.0 / np.sqrt(loss.item()))
print(f" PSNR rekonstrukcji: {psnr:.1f} dB")
print("\n Na OAI Inpainting trzeba było uzupełnić brakujące fragmenty obrazu używając INR!")